## Train a Variational Autoencoder using Galaxies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
import numpy as np
from torchvision.datasets import CIFAR10
from torchvision.utils import make_grid, save_image
from torchvision import datasets, transforms
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm, trange
import os
import h5py
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Prepare the Data

In [ ]:
with h5py.File('Galaxy10_DECals.h5', 'r') as gF:
    images = np.array(gF['images'])

images = shuffle(images)
images = torch.Tensor(images).permute(0, 3, 1, 2)  # Convert to torch.Tensor and change to (N, C, H, W)
images = F.interpolate(images, size=(32, 32), mode='bilinear', align_corners=False)  # Downsample

train_idx, test_idx = train_test_split(np.arange(images.shape[0]), test_size=0.1)
X_train, X_test = images[train_idx], images[test_idx]
X_train /= 255
X_test /= 255

train_dataset = TensorDataset(X_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, 
                                          shuffle=True, drop_last=True)

test_dataset = TensorDataset(X_test)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, 
                                          shuffle=False, drop_last=True)

In [ ]:
def show_tensor_images(image_tensor, 
                       num_images=64, 
                       size=(3, 32, 32), 
                       nrow=8, 
                       filename=""):
    image_unflat = image_tensor.detach().cpu().view(-1, *size)
    image_grid = make_grid(image_unflat[:num_images], nrow=nrow)
    img = image_grid.permute(1, 2, 0).squeeze()
    plt.axis('off')
    plt.imshow(img)
    if len(filename) > 0:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### Build the Autoencoder

In [ ]:
class Sampling(nn.Module):
    def forward(self, inputs):
        mu, log_var = inputs
        std = torch.exp(0.5 * log_var)
        epsilon = torch.randn_like(std)
        return mu + std * epsilon

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Encoder, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=input_dim[0], out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(6400, 256)  # Adjusted for CIFAR-10 dimensions
        self.mu = nn.Linear(256, latent_dim)
        self.log_var = nn.Linear(256, latent_dim)
        self.sampling = Sampling()

    def forward(self, x):
        x = F.silu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, kernel_size=2, padding=1)
        x = F.silu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, kernel_size=2, padding=1)
        x = F.silu(self.bn3(self.conv3(x)))
        x = F.max_pool2d(x, kernel_size=2, padding=1)
        x = self.flatten(x)
        x = F.relu(self.fc(x))
        mu = self.mu(x)
        log_var = self.log_var(x)
        z = self.sampling((mu, log_var))
        return mu, log_var, z

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super(Decoder, self).__init__()
        self.fc = nn.Linear(latent_dim, 4 * 4 * 128)
        self.deconv1 = nn.ConvTranspose2d(128, 128, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.bn1 = nn.BatchNorm2d(128)
        self.deconv2 = nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.deconv3 = nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.bn3 = nn.BatchNorm2d(32)
        self.deconv4 = nn.ConvTranspose2d(32, 3, kernel_size=3, padding=1)
    
    def forward(self, x):
        x = F.relu(self.fc(x))
        x = x.view(-1, 128, 4, 4)  # Reshaping to start the upsampling process
        x = F.silu(self.bn1(self.deconv1(x)))  # Upsample to 8x8
        x = F.silu(self.bn2(self.deconv2(x)))  # Upsample to 16x16
        x = F.silu(self.bn3(self.deconv3(x)))  # Upsample to 32x32
        x = torch.sigmoid(self.deconv4(x))  # Final output to 32x32 with 3 channels
        return x

In [ ]:
class VAE(nn.Module):
    def __init__(self, encoder, decoder, beta=1.0):
        super(VAE, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta

    def forward(self, x):
        mu, log_var, z = self.encoder(x)
        reconstruction = self.decoder(z)
        return reconstruction, mu, log_var

In [ ]:
input_dim = (3, 32, 32)
latent_dim = 128

In [ ]:
encoder = Encoder(input_dim=input_dim, latent_dim=latent_dim).to(device)
decoder = Decoder(latent_dim=latent_dim).to(device)
vae = VAE(encoder, decoder).to(device)

In [ ]:
learning_rate = 0.001
optimizer = torch.optim.Adam(vae.parameters(), lr=learning_rate)

In [ ]:
num_epochs = 200
beta = 1
epoch_freq = 50
img_batch = 32768
blocks = 64

In [ ]:
def vae_loss(recon_x, x, mean, log_var, beta):
    total_elements = recon_x.numel()
    recon_loss = F.binary_cross_entropy(recon_x.view(-1, total_elements), x.view(-1, total_elements), reduction='sum')
    kld_loss = -0.5 * torch.sum(1 + log_var - mean ** 2 - log_var.exp())
    return recon_loss * beta + kld_loss, recon_loss, kld_loss

In [ ]:
vaecifar10_filepath = './VAEGalaxies'
if not os.path.exists(vaecifar10_filepath):
    os.makedirs(vaecifar10_filepath)

In [ ]:
loss_lst = list()
reloss_list = list()
kl_loss_list = list()
tqdm_epoch = trange(num_epochs)

for epoch in tqdm_epoch:
    sum_loss = 0.0
    num_items = 0
    re_loss = 0.0
    kl_loss = 0.0
    for [data] in train_loader:
        data = data.to(device)
        cur_batch_size = len(data)
        optimizer.zero_grad()
        recon_batch, mean, log_var = vae(data)
        loss, rc_loss, kl_loss = vae_loss(recon_batch, data, mean, log_var, beta)
        loss.backward()
        optimizer.step()
        
        sum_loss += loss.item() * cur_batch_size
        re_loss += rc_loss.item() * cur_batch_size
        kl_loss += kl_loss.item() * cur_batch_size
        num_items += cur_batch_size
    
    ave_loss = sum_loss / num_items
    ave_re_loss = re_loss / num_items
    ave_kl_loss = kl_loss / num_items
    loss_lst.append(ave_loss)
    tqdm_epoch.set_description('Ave Loss: {:5f}, Ave Rec Loss: {:5f},'
                               'Ave KL Loss: {:5f}'.format(ave_loss, ave_re_loss, ave_kl_loss))
    
    if (epoch + 1) % epoch_freq == 0:
        epoch_path = os.path.join(vaecifar10_filepath, f"epoch_{epoch + 1}")
        if not os.path.exists(epoch_path):
            os.makedirs(epoch_path)
        block_size = img_batch // blocks
        for iblk in range(blocks):
            idx_start = iblk * block_size
            idx_end = (iblk + 1) * block_size
            sample_z = torch.randn(block_size, latent_dim).to(device)
            sample_imgs = vae.decoder(sample_z)
            
            for idx in range(idx_start, idx_end):
                filename = os.path.join(epoch_path, f"generation_{idx}.png")
                save_image(sample_imgs[idx - idx_start], filename)


In [ ]:
torch.save(vae.state_dict(), 'vaegalaxies_genckpt.pth')

### Display Reconstructions

In [ ]:
test_z = encoder(X_test[:20].to(device))
reconstruction = decoder(test_z[2])

In [ ]:
plt.figure(figsize=(10, 20))
show_tensor_images(reconstruction, 
                       num_images=20, 
                       size=(3, 32, 32), 
                       nrow=20, 
                       filename="vaegalaxiesreconstruct.png")

In [ ]:
plt.figure(figsize=(10, 20))
show_tensor_images(X_test,
                   num_images=20, 
                   size=(3, 32, 32), 
                   nrow=20,  
                   filename='vaegalaxiestest.png')

### Generate samples

In [ ]:
sample_z = torch.randn(64, latent_dim).to(device)

In [ ]:
decoded_samples = vae.decoder(sample_z)

In [ ]:
plt.figure(figsize=(12, 12))
show_tensor_images(decoded_samples, 
                       num_images=64, 
                       size=(3, 32, 32), 
                       nrow=8, 
                       filename="VAEGalaxiesGenerationGrid.png")